# News Article Categorizer - DistilBERT
Run: Runtime > Change runtime type > T4 GPU, then Runtime > Run all.

In [1]:
!pip install -q transformers

In [7]:
import urllib.request, os, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

BASE = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/"
os.makedirs("data", exist_ok=True)
for f in ["train.csv", "test.csv"]:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve(BASE + f, "data/" + f)

N_TRAIN, MAX_LEN, MODEL = 20000, 128, "distilbert-base-uncased"

def load(path):
    d = pd.read_csv(path, header=None, names=["label", "title", "desc"])
    d["text"] = d.title.str.strip() + ". " + d.desc.str.replace("\\", " ", regex=False).str.strip()
    d["label"] = d.label - 1                      # labels 1-4 -> 0-3
    return d[["text", "label"]]

train = load("data/train.csv").sample(N_TRAIN, random_state=42)
test = load("data/test.csv")

tok = AutoTokenizer.from_pretrained(MODEL)
class DS(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tok(df.text.tolist(), truncation=True, padding=True, max_length=MAX_LEN)
        self.y = df.label.tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.y[i])
        return item

def metrics(p):
    pred = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, pred),
            "macro_f1": f1_score(p.label_ids, pred, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=4)
args = TrainingArguments(output_dir="distilbert_out", num_train_epochs=2,
                         per_device_train_batch_size=32, per_device_eval_batch_size=64,
                         learning_rate=2e-5, weight_decay=0.01, logging_steps=100,
                         save_strategy="no", report_to="none",
                         fp16=torch.cuda.is_available())
trainer = Trainer(model=model, args=args, train_dataset=DS(train),
                  eval_dataset=DS(test), compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.647279
200,0.324260
300,0.311618
400,0.291434
500,0.242136
600,0.239594
700,0.207586
800,0.173154
900,0.182880
1000,0.211080


Training Loss,Validation Loss,Step,Accuracy,Macro F1
0.160955,0.232285,1250,0.921974,0.921952


{'eval_loss': 0.23228473961353302, 'eval_accuracy': 0.9219736842105263, 'eval_macro_f1': 0.9219523330463195}
